In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from sklearn import metrics

# 1. EDA

In [ ]:
df = pd.read_csv("1 - Project Data.csv")

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df['Churn Value'].value_counts(normalize=True)

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.shape

# 2. Cleaning

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce').fillna(0.0)

In [ ]:
addons = ['Online Security', 'Online Backup', 'Device Protection',
          'Tech Support', 'Streaming TV', 'Streaming Movies']
df_clean[addons] = df_clean[addons].replace('No internet service', 'No')
df_clean['Multiple Lines'] = df_clean['Multiple Lines'].replace('No phone service', 'No')

# 3. Feature Engineering

In [ ]:
DROPS = [
    "Count", "Country", "State",
    "CustomerID",
    "Churn Label", "Churn Value",
    "Churn Reason",
    "Lat Long",
    "City", "Zip Code", "Latitude", "Longitude",
    "Total Charges",
]

customer_ids = df_clean["CustomerID"]
y = df_clean["Churn Value"]
X = df_clean.drop(columns=DROPS)

In [ ]:
X_enc = pd.get_dummies(X, drop_first=True, dtype=int)
FEATURE_COLUMNS = list(X_enc.columns)

In [ ]:
X_enc.head()

# 4. Scaling

In [ ]:
NUMERIC = ["Tenure Months", "Monthly Charges"]

scaler = StandardScaler()
X_enc[NUMERIC] = scaler.fit_transform(X_enc[NUMERIC])

In [ ]:
X_enc[NUMERIC].describe()

# 5. Fitting the Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
results = X_test.copy()
results[['prob_stay', 'prob_churn']] = model.predict_proba(X_test)
results['y_pred'] = np.where(results['prob_churn'] > .5, 1, 0)

roc_auc_score(y_test, results['prob_churn'])

In [ ]:
results.head()

# 6. Metrics

In [ ]:
def produce_confusion(positive_label, negative_label, cut_off, df, y_pred_name, y_real_name):
    pred = df[y_pred_name] if cut_off == 'binary' else np.where(df[y_pred_name] > cut_off, 1, 0)

    cm = confusion_matrix(df[y_real_name], pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, ax=ax, fmt='g', cmap='Blues', cbar=False)

    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('Real labels')
    ax.set_title('Confusion Matrix')
    ax.xaxis.set_ticklabels([negative_label, positive_label])
    ax.yaxis.set_ticklabels([negative_label, positive_label])
    plt.show()

    acc = accuracy_score(df[y_real_name], pred)
    print('Test accuracy = ', acc)
    return acc

In [ ]:
results['y_real'] = y_test
produce_confusion('Churned', 'Retained', 'binary', results, 'y_pred', 'y_real')

In [ ]:
print(metrics.classification_report(y_test, results['y_pred']))

# 7. Finding the Best Threshold Value

In [ ]:
for cut in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    pred = np.where(results['prob_churn'] > cut, 1, 0)
    print(f"{cut:.2f}  acc {metrics.accuracy_score(y_test, pred):.3f}"
          f"  recall {metrics.recall_score(y_test, pred):.3f}"
          f"  precision {metrics.precision_score(y_test, pred):.3f}"
          f"  f1 {metrics.f1_score(y_test, pred):.3f}")

# 8. The 500 Customers Most Likely to Churn

In [ ]:
cust_retain = df[df['Churn Value'] == 0]["CustomerID"].copy()

final = pd.concat([customer_ids, X_enc], axis=1)
final[['prob_stay', 'prob_churn']] = model.predict_proba(X_enc)
final['y_pred'] = np.where(final['prob_churn'] > .5, 1, 0)

final = final[final["CustomerID"].isin(cust_retain)]

final_sorted = final.sort_values(by="prob_churn", ascending=False).copy()

five_hund_churn = final_sorted.head(500)
five_hund_churn

In [ ]:
scored = df_clean[["CustomerID", "Churn Value", "City", "Zip Code",
                   "Tenure Months", "Contract", "Internet Service",
                   'Online Security', 'Online Backup', 'Device Protection',
                   'Tech Support', 'Streaming TV',
                   'Streaming Movies', "Payment Method",
                   "Monthly Charges", "Total Charges"]].copy()

scored[["prob_stay", "prob_churn"]] = model.predict_proba(X_enc)

retained = scored[scored["Churn Value"] == 0].drop(columns="Churn Value").copy()
mailer_list = retained.sort_values("prob_churn", ascending=False).head(500)
mailer_list

# 9. Average churn risk per service

In [ ]:
addons

In [ ]:
retained_online_sec = retained[retained["Online Security"] == "Yes"].copy()
retained_online_bckup = retained[retained["Online Backup"] == "Yes"].copy()
retained_dev_prot = retained[retained["Device Protection"] == "Yes"].copy()
retained_tech_sup = retained[retained["Tech Support"] == "Yes"].copy()
retained_stream_tv = retained[retained["Streaming TV"] == "Yes"].copy()
retained_stream_movie = retained[retained["Streaming Movies"] == "Yes"].copy()
addon_retained_dfs = {'Online Security': retained_online_sec,
 'Online Backup': retained_online_bckup,
 'Device Protection': retained_dev_prot,
 'Tech Support': retained_tech_sup,
 'Streaming TV': retained_stream_tv,
 'Streaming Movies': retained_stream_movie}

In [ ]:
for feature in addons:
    sign_up_feature = addon_retained_dfs[feature]
    sign_up_feature = sign_up_feature.sort_values(by="prob_churn", ascending=False).head(500)
    addon_retained_dfs[feature] = sign_up_feature

In [ ]:
sign_up_feature_churn_risk = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}
for feature in addons:
    sign_up_feature_churn_risk[feature] = float(np.round(addon_retained_dfs[feature]["prob_churn"].mean(), 4))

In [ ]:
sign_up_feature_churn_risk

# 10. Revenue lost per feature
The amount of lost revenue for users that have already churned sum of monthly charges for fibre specifically

In [ ]:
churned = scored[scored["Churn Value"] == 1].drop(columns="Churn Value").copy()

In [ ]:
churned_online_sec = churned[churned["Online Security"] == "Yes"].copy()
churned_online_bckup = churned[churned["Online Backup"] == "Yes"].copy()
churned_dev_prot = churned[churned["Device Protection"] == "Yes"].copy()
churned_tech_sup = churned[churned["Tech Support"] == "Yes"].copy()
churned_stream_tv = churned[churned["Streaming TV"] == "Yes"].copy()
churned_stream_movie = churned[churned["Streaming Movies"] == "Yes"].copy()
addon_churned_dfs = {'Online Security': churned_online_sec,
 'Online Backup': churned_online_bckup,
 'Device Protection': churned_dev_prot,
 'Tech Support': churned_tech_sup,
 'Streaming TV': churned_stream_tv,
 'Streaming Movies': churned_stream_movie}

In [ ]:
sign_up_feature_total_rev_lost = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}
sign_up_feature_monthly_rev_lost = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}
for feature in addons:
    sign_up_feature_monthly_rev_lost[feature] = float(np.round(addon_churned_dfs[feature]["Monthly Charges"].sum(), 4))
    sign_up_feature_total_rev_lost[feature] = float(np.round(addon_churned_dfs[feature]["Total Charges"].sum(), 4))

In [ ]:
sign_up_feature_monthly_rev_lost

In [ ]:
sign_up_feature_total_rev_lost

# 11. Monthly cost per feature

In [ ]:
total = scored.drop(columns="Churn Value").copy()

In [ ]:
online_sec = total[total["Online Security"] == "Yes"].copy()
online_bckup = total[total["Online Backup"] == "Yes"].copy()
dev_prot = total[total["Device Protection"] == "Yes"].copy()
tech_sup = total[total["Tech Support"] == "Yes"].copy()
stream_tv = total[total["Streaming TV"] == "Yes"].copy()
stream_movie = total[total["Streaming Movies"] == "Yes"].copy()
addon_dfs = {'Online Security': online_sec,
 'Online Backup': online_bckup,
 'Device Protection': dev_prot,
 'Tech Support': tech_sup,
 'Streaming TV': stream_tv,
 'Streaming Movies': stream_movie}

In [ ]:
sign_up_feature_monthly_charge = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}

In [ ]:
for feature in addons:
    sign_up_feature_monthly_charge[feature] = float(np.round(addon_dfs[feature]["Monthly Charges"].mean(), 4))

In [ ]:
sign_up_feature_monthly_charge

# 12. Compare monthly charges for churned vs retained users

In [ ]:
retained_sign_up_feature_monthly_charge = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}

churned_sign_up_feature_monthly_charge = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}

In [ ]:
for feature in addons:
    retained_sign_up_feature_monthly_charge[feature] = float(np.round(addon_retained_dfs[feature]["Monthly Charges"].mean(), 4))
for feature in addons:
    churned_sign_up_feature_monthly_charge[feature] = float(np.round(addon_churned_dfs[feature]["Monthly Charges"].mean(), 4))

In [ ]:
retained_sign_up_feature_monthly_charge

In [ ]:
churned_sign_up_feature_monthly_charge

# 13. Revenue kept

In [ ]:
sign_up_feature_total_rev_kept = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}
sign_up_feature_monthly_rev_kept = {'Online Security': 0,
 'Online Backup': 0,
 'Device Protection': 0,
 'Tech Support': 0,
 'Streaming TV': 0,
 'Streaming Movies':0}
for feature in addons:
    sign_up_feature_monthly_rev_kept[feature] = float(np.round(addon_retained_dfs[feature]["Monthly Charges"].sum(), 4))
    sign_up_feature_total_rev_kept[feature] = float(np.round(addon_retained_dfs[feature]["Total Charges"].sum(), 4))


In [ ]:
sign_up_feature_monthly_rev_kept

In [ ]:
sign_up_feature_total_rev_kept